In [ ]:
!wget "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [2]:
import os
import shutil
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
from numpy.typing import NDArray
import random

load_dotenv()

# 로컬
# ROOT = Path(os.environ["DATA_ROOT"])
# HF_HOME = ROOT / ".hf_cache"
# os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_HOME"] = ".hf_cache"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import f1_score, label_ranking_average_precision_score

In [3]:
# Config
config = {
    "seed": 42,
    "num_labels": 188, "max_len": 8192, "eff_batch": 8,
    "ckpt": "ingyoun/A.X-patent-maxlen8192",
    "clean_ds": "ingyoun/patent-clean-text",
    "tokenized_ds": "ingyoun/patent-clean-text-modernbert-tokenized",
    "cache_dir": "/content/drive/MyDrive/.hf_cache",
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],   # 기록용
    'out_path': '/content/output/',                                    # colab
    # "out_path": ROOT / "output",                                     # 로컬
    "tag": "modernbert-patent-len8192",                            # f"{model}_len{max_len}"
}

In [4]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [5]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cuda


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Inference

In [7]:
class EvalCollator:
    """동적 패딩"""
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, feats):
        enc = [
            {"input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"]}
            for f in feats
        ]
        return self.tok.pad(enc, padding=True, return_tensors="pt")


class LogitsRunner:
    """split별 logits를 추론.캐시"""
    def __init__(self, model, tokenizer, cache_dir, tag, batch_size=8):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(self.device).eval()
        self.collate = EvalCollator(tokenizer)
        self.cache_dir, self.tag, self.batch_size = Path(cache_dir), tag, batch_size

    @torch.no_grad
    def _infer(self, ds):
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)
        chunks = []
        for enc in loader:
            enc = {k: v.to(self.device) for k, v in enc.items()}
            if self.device == "cuda":
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = self.model(**enc).logits
            else:
                logits = self.model(**enc).logits
            chunks.append(logits.float().cpu().numpy())
        return np.concatenate(chunks, axis=0)

    def get(self, ds, split):
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        fp = self.cache_dir / f"logits_{self.tag}_{split}.npy"
        if fp.exists():
            return np.load(fp)
        arr = self._infer(ds)
        np.save(fp, arr)
        return arr

## 평가 지표

In [8]:
class LengthBinner:
    """
    문서별 kobert_len -> 고정 4구간 bin별로 평가.
    truncation에 따른 정보 손실로 실제 모델 성능에 영향이 미치는지 확인하기 위함
    """
    EDGEs = [512, 1024, 2048]
    LABELS = ["B0 (≤512)", "B1 (512–1024)", "B2 (1024–2048)", "B3 (>2048)"]
    def __init__(self, bins, labels=None, edges=None):
        self.bins = bins
        self.labels = labels or self.LABELS
        self.edges = edges or self.EDGEs

    @classmethod
    def from_hub(cls, dataset:str, doc_ids, edges=None):
        edges = edges or cls.EDGEs
        ds = load_dataset(dataset, cache_dir=config["cache_dir"] , split="test")
        length_of = dict(zip(ds["document_id"], ds["kobert_len"]))
        lengths = np.array([length_of[d] for d in doc_ids])
        return cls(np.digitize(lengths, edges, right=True), edges=edges)

    def masks(self):
        for b, label in enumerate(self.labels):
            yield b, label, self.bins == b

In [9]:
class MultiLabelEvaluator:
    """test 예측·정답·로짓 + LengthBinner → 멀티라벨 지표 dict.

    벤더 지표의 top-1 붕괴(멀티라벨을 top-1 단일라벨로 접는 문제)를 바로잡아,
    같은 고정 test split 위에서 진짜 멀티라벨 지표를 산출한다.

    Parameters
    ----------
    P : NDArray[float], shape [N, C=188]
        예측 확률. ``sigmoid(logits)`` 결과라 열마다 독립 확률(∈[0,1]).
        행 i = 문서 i, 열 c = "문서 i가 Mno id c에 속할 확률".
        softmax가 아니므로 **행 합이 1이 아니다**(멀티라벨).
    Y : NDArray[int], shape [N, C=188]
        gold 멀티핫. ``Y[i, c] == 1`` 이면 문서 i가 라벨 c를 가진다.
        본 데이터는 문서당 평균 ~1.2 라벨이라 대부분의 행이 1을 하나만 갖는다.
    logits : NDArray[float], shape [N, C=188]
        sigmoid 적용 **전** 원본 로짓. top-1 앵커가 이 로짓의 argmax를 쓴다
        벤더 재현 코드와 입력 형식을 맞추기 위해 로짓을 그대로 보관한다.
    binner : LengthBinner
        문서별 길이 bin 마스크(``masks()``)를 제공.
        장문 가설 검증(길이구간별 F1)의 축.
        bin은 KoBERT ``kobert_len`` 기준으로 고정된다.
    """
    def __init__(self, P, Y, logits, binner):
        self.P = P
        self.Y = Y
        self.logits = logits
        self.binner = binner

    def _predict(self, P, force_argmax: bool = False) -> NDArray[np.integer]:
        """ 확률 0/1 예측. 고정 tau=0.5 """
        pred = (P >= 0.5).astype(int)
        if force_argmax:                              # 빈 예측이 나오면 가장 확신 높은 라벨 강제 부여
            empty = pred.sum(1) == 0                  # 라벨 별로 positive를 확인
            if empty.any():                           # 빈 예측인지 확인
                pred[empty, P[empty].argmax(1)] = 1   # 빈 예측 문서에 강제로 예측 라벨 부여
        return pred

    def multilabel_f1(self, pred):
        return {
            name: f1_score(self.Y, pred, average=avg, zero_division=0)
            for name, avg in (("micro", "micro"), ("macro", "macro"), ("sample", "samples"))
        }

    def length_bins(self, which="micro"):
        rows = []
        for _, label, mask in self.binner.masks():      # binner.masks : (bin 번호, bin 라벨, bin에 대한 mask)
            row = {"bin": label, "n": int(mask.sum())}
            if mask.any():
                pred = self._predict(self.P[mask])      # bin에 속하는 문서들만 골라서 확률 계산
                row.update({
                    avg: f1_score(self.Y[mask], pred, average=avg, zero_division=0) for avg in ("micro", "macro")
                })
            rows.append(row)
        return rows

    def ranking(self):
        order = np.argsort(-self.P, axis=1)            # 문서마다 독립적으로 확률 높은 라벨 순서로 내림차순 정렬
        n = self.Y.sum(1).astype(int)                  # 문서마다 정답 라벨 개수
        rprec = np.mean([self.Y[i, order[i, :max(n[i], 1)]].sum() / max(n[i], 1) for i in range(len(self.Y))])  # 정답이 n개인 문서라면, 모델이 상위 n개로 꼽은 것 중 몇 개가 실제 정답인가
        return {
            "lrap": float(label_ranking_average_precision_score(self.Y, self.P)),
            "r_precision": float(rprec)
        }

    def anchor_top1(self):                                     # 벤더 baseline과의 연속성
        pred1, gold1 = self.logits.argmax(1), self.Y.argmax(1)
        out = {
            f"{avg}_f1": f1_score(gold1, pred1, average=avg, zero_division=0) for avg in ("weighted", "micro", "macro")
        }

        order = np.argsort(-self.logits, axis=1)                  # 내림차순 정렬 인덱스. 문서마다 확신 순 라벨 나열
        for k in (1, 3, 5):
            hit = np.take_along_axis(self.Y, order[:, :k], axis=1).sum(1)
            out[f"p@{k}"] = float((hit / np.clip(self.Y.sum(1), 1, k)).mean())
        return out

    def evaluate(self) -> dict:
        pred_keep = self._predict(self.P)                         # tau = 0.5, 빈 예측 허용
        pred_argmax = self._predict(self.P, force_argmax=True)    # 빈 예측 -> argmax 1개 강제
        return {
            "test_n": int(len(self.Y)),
            "tau": {"micro": 0.5, "note": "fixed value(default)"},
            "empty_rate_tau_micro": float((pred_keep.sum(1) == 0).mean()),
            "multilabel_f1": {
                "keep": self.multilabel_f1(pred_keep),            # headline 메인 지표
                "argmax": self.multilabel_f1(pred_argmax),        # 빈 예측 보정
            },
            "length_bins": self.length_bins("micro"),
            "ranking": self.ranking(),
            "anchor_top1": self.anchor_top1()
        }

## 보고

In [10]:
class EvalReport:
    def __init__(self, result: dict, meta: dict):
        self.result = {**meta, **result}            # meat : model/fields/max_length/tag

    def to_json(self, out_path):
        out_path = Path(out_path)
        out_path.mkdir(parents=True, exist_ok=True)
        fp = out_path / f"total_metrics_{self.result['tag']}.json"
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(self.result, f, ensure_ascii=False, indent=2)
        return fp

    def to_markdown(self):
        r = self.result
        keep = r["multilabel_f1"]["keep"]
        lines = [
            f"### {r['model']} (test n={r['test_n']})",
            f"- F1 @τ={r['tau']['micro']}: "
            f"{keep['micro']:.4f} / {keep['macro']:.4f} / {keep['sample']:.4f}",
            f"- anchor top-1 weighted-F1: {r['anchor_top1']['weighted_f1']:.4f}",
            f"- LRAP / R-Precision: {r['ranking']['lrap']:.4f} / {r['ranking']['r_precision']:.4f}",
            f"- empty rate: {r['empty_rate_tau_micro']:.3f}",
            "",
            "| bin | n | micro | macro |",
            "| --- | --- | --- | --- |",
        ]
        for b in r["length_bins"]:
            lines.append(
                f"| {b['bin']} | {b['n']} | "
                f"{b.get('micro', float('nan')):.4f} | {b.get('macro', float('nan')):.4f} |"
            )
        return "\n".join(lines)

## Ocherstrator

In [11]:
class EvaluationHarness:
    """ config 객체 조립 -> EvalReport """
    def __init__(self, config: dict):
        self.cfg = config

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    @staticmethod
    def _truncate(ds, tokenizer, max_len):
        """훈련(_prep)과 동일하게 max_len으로 절단 — <s> 유지 + 꼬리를 <\\s>(eos)로 마감.
        토큰화 데이터셋은 절단 없이 전체 토큰이라, 훈련 창과 맞춰야 정확한 평가가 된다."""
        eos_id = tokenizer.eos_token_id
        def _fn(batch):
            ids, masks = [], []
            for x, m in zip(batch["input_ids"], batch["attention_mask"]):
                if len(x) > max_len:
                    x = x[: max_len - 1] + [eos_id]
                    m = m[:max_len]
                ids.append(x)
                masks.append(m)
            return {"input_ids": ids, "attention_mask": masks}
        return ds.map(_fn, batched=True)

    def run(self) -> "EvalReport":
        cfg = self.cfg
        model = AutoModelForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path=config["ckpt"],
            dtype=torch.float32,
            attn_implementation="flash_attention_2",
        )
        tokenizer = AutoTokenizer.from_pretrained(config["ckpt"])
        test_ds = load_dataset(
            config["tokenized_ds"],
            cache_dir=config["cache_dir"],
            split="test"
        ) # [11271, 188]
        test_ds = self._truncate(test_ds, tokenizer, cfg["max_len"])   # 훈련과 동일 창으로 절단

        # 추론
        runner = LogitsRunner(model=model, tokenizer=tokenizer, cache_dir=cfg["out_path"], tag=cfg["tag"], batch_size=cfg["eff_batch"])
        test_logits = runner.get(test_ds, "test")
        P = self._sigmoid(test_logits)
        Y = np.asarray(test_ds["labels"], dtype=int)
        # 길이 bin
        binner = LengthBinner.from_hub(cfg["tokenized_ds"], test_ds["document_id"])
        # 지표 산출
        evaluator = MultiLabelEvaluator(P, Y, test_logits, binner)
        meta = {
            "model": cfg["ckpt"],
            "fields": cfg["fields"],
            "max_length": cfg["max_len"],
            "tag": cfg["tag"]
        }
        return EvalReport(evaluator.evaluate(), meta)


## 실행

In [ ]:
report = EvaluationHarness(config=config).run()
fp = report.to_json(config["out_path"])

In [13]:
shutil.copy(fp, "/content/drive/MyDrive/patent_disc/")

'/content/drive/MyDrive/patent_disc/total_metrics_modernbert-patent-len8192.json'

In [14]:
print(report.to_markdown())

### ingyoun/A.X-patent-maxlen8192 (test n=11271)
- F1 @τ=0.5: 0.8685 / 0.8649 / 0.8825
- anchor top-1 weighted-F1: 0.8256
- LRAP / R-Precision: 0.9371 / 0.8970
- empty rate: 0.013

| bin | n | micro | macro |
| --- | --- | --- | --- |
| B0 (≤512) | 3197 | 0.8765 | 0.8662 |
| B1 (512–1024) | 5183 | 0.8734 | 0.8696 |
| B2 (1024–2048) | 2342 | 0.8516 | 0.8398 |
| B3 (>2048) | 549 | 0.8490 | 0.7516 |
